In [4]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# Load data
df = pd.read_csv('../data/all_seasons.csv')

# Keep only the columns that actually exist in your CSV
cols = ['player_name','season','age','player_height','player_weight',
        'pts','reb','ast','net_rating','usg_pct','ts_pct',
        'draft_round','gp','team_abbreviation']
df = df[cols].copy()

# Filter low game count instead of low minutes (gp = games played)
df = df[df['gp'] >= 20].copy()

# Drop missing values
df = df.dropna(subset=['net_rating','pts','reb','ast','usg_pct','ts_pct'])

# Clean draft_round
df['draft_round'] = df['draft_round'].replace('Undrafted', '3')
df['draft_round'] = pd.to_numeric(df['draft_round'], errors='coerce').fillna(3)

# Create engineered features
df['log_pts'] = np.log1p(df['pts'])
df['pts_usg'] = df['pts'] * df['usg_pct']

print(f"Dataset shape: {df.shape}")
print(f"\nnet_rating summary:")
print(df['net_rating'].describe().round(2))

Dataset shape: (10720, 16)

net_rating summary:
count    10720.00
mean        -1.09
std          6.44
min        -40.00
25%         -5.30
50%         -0.80
75%          3.20
max         19.50
Name: net_rating, dtype: float64


In [5]:
features_full = ['age','pts','reb','ast','usg_pct','ts_pct','draft_round']
features_domain = ['pts','ast','reb','usg_pct']
features_transformed = ['log_pts','ast','reb','usg_pct','pts_usg']
target = 'net_rating'

X_full = df[features_full]
X_domain = df[features_domain]
X_transformed = df[features_transformed]
y = df[target]

X_train_f, X_test_f, y_train, y_test = train_test_split(X_full, y, test_size=0.2, random_state=42)
X_train_d, X_test_d, _, _ = train_test_split(X_domain, y, test_size=0.2, random_state=42)
X_train_t, X_test_t, _, _ = train_test_split(X_transformed, y, test_size=0.2, random_state=42)

print("Split done. Training rows:", len(y_train))

Split done. Training rows: 8576


In [7]:
def fit_ols(X_train, y_train, model_name):
    """Fit OLS with statsmodels and print summary."""
    X_const = sm.add_constant(X_train)
    model = sm.OLS(y_train, X_const).fit()
    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print('='*60)
    print(model.summary())
    return model

model1 = fit_ols(X_train_f, y_train, "MODEL 1 — Full Model")
model2 = fit_ols(X_train_d, y_train, "MODEL 2 — Domain-Driven Model")
model3 = fit_ols(X_train_t, y_train, "MODEL 3 — Transformed Model")


  MODEL 1 — Full Model
                            OLS Regression Results                            
Dep. Variable:             net_rating   R-squared:                       0.204
Model:                            OLS   Adj. R-squared:                  0.203
Method:                 Least Squares   F-statistic:                     313.0
Date:                Thu, 02 Apr 2026   Prob (F-statistic):               0.00
Time:                        00:10:02   Log-Likelihood:                -27120.
No. Observations:                8576   AIC:                         5.426e+04
Df Residuals:                    8568   BIC:                         5.431e+04
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
const         -23.2165    

## Model 1 — Full Model: Observations

**R² = 0.204 | Adj R² = 0.203 | AIC = 54,260**

Model 1 includes all seven predictors as a baseline. It achieves the highest R² of the three models, meaning it explains approximately 20% of the variation in net_rating. While this may seem low, it is expected for individual player data — net_rating is heavily influenced by teammates, coaching systems, and opponent strength, none of which are captured by personal statistics alone.

**Key findings from the coefficients:**

- **ts_pct (coef = 30.37, p < 0.001):** True shooting percentage is the strongest predictor in this model. A 1-unit increase in shooting efficiency adds 30.37 points to net_rating, confirming that efficiency matters far more than volume in determining a player's impact.
- **ast (coef = 0.41, p < 0.001):** Each additional assist per game adds 0.41 to net_rating. Assists reflect ball movement and playmaking, which directly improve team offensive flow.
- **usg_pct (coef = −15.64, p < 0.001):** Higher usage rate is associated with lower net_rating. This is because high-usage players are often on weaker teams that rely on one player to do everything, or because heavy usage comes at a cost to team efficiency.
- **pts (coef = 0.15, p < 0.001):** Points per game has a small positive effect, but it is weaker than expected because scoring volume without efficiency (captured by ts_pct) does not reliably improve team performance.
- **age (coef = 0.24, p < 0.001):** Older players tend to have slightly higher net_ratings, likely because veterans earn roles on better-structured teams and play more efficient minutes.
- **reb (coef = 0.18, p < 0.001):** Rebounds contribute positively to net_rating, reflecting their role in controlling possession.
- **draft_round (coef = −0.22, p = 0.013):** Later draft picks have slightly lower net_ratings on average, consistent with the idea that higher-drafted players are placed in better team environments.

**Limitation:** The condition number is 1,140, which indicates multicollinearity — some predictors (particularly pts and usg_pct) are correlated with each other. This inflates standard errors and makes individual coefficients less reliable. This model is useful as a baseline but not selected as the final model.

## Model 2 — Domain-Driven Model: Observations

**R² = 0.124 | Adj R² = 0.123 | AIC = 55,070**

Model 2 retains only the four core NBA performance metrics: points, assists, rebounds, and usage rate. These were selected based on basketball domain knowledge — they are the statistics most commonly used by analysts to evaluate player contribution to team success. Age, true shooting percentage, and draft round were excluded as they are contextual variables rather than direct performance measures.

The R² drops to 0.124 compared to Model 1, which is expected since we removed two significant predictors (ts_pct and age). However, this model is cleaner — there is no multicollinearity warning, and all four predictors are statistically significant (all p < 0.001).

**Key findings from the coefficients:**

- **pts (coef = 0.44, p < 0.001):** Points per game is now the strongest predictor. Each additional point per game adds 0.44 to net_rating. Without ts_pct in the model, pts absorbs more of the scoring effect.
- **usg_pct (coef = −39.06, p < 0.001):** Usage rate has a strongly negative effect. A player who dominates ball possession without proportional efficiency significantly reduces team net_rating. This is the most important finding in this model.
- **ast (coef = 0.28, p < 0.001):** Assists remain a consistent positive predictor across models, confirming their genuine impact on team performance.
- **reb (coef = 0.19, p < 0.001):** Rebounds contribute positively and consistently, reflecting the importance of possession control.

**Limitation:** By removing ts_pct, this model loses the efficiency dimension of scoring. A player who scores 20 points on poor shooting is treated the same as one who scores 20 points efficiently. This is a meaningful omission that Model 3 addresses through its log transformation and interaction term.

## Model 3 — Transformed Model: Observations

**R² = 0.128 | Adj R² = 0.127 | AIC = 55,030**

Model 3 is the most theoretically motivated model. It replaces the raw points variable with log(pts + 1) to capture diminishing returns to scoring, and adds an interaction term (pts × usg_pct) to capture conditional scoring efficiency. This reflects a key insight from basketball economics: a player scoring 10 points on a weak team contributes differently than one scoring 10 points as a secondary option on a strong team.

The R² of 0.128 is slightly higher than Model 2, and the AIC of 55,030 is the lowest among Models 2 and 3, indicating a better balance between fit and model complexity. Crucially, the condition number is only 242 — the lowest of all three models — meaning multicollinearity is not a concern here.

**Key findings from the coefficients:**

- **log_pts (coef = 2.51, p < 0.001):** The log-transformed points variable is the strongest predictor. A 1-unit increase in log(pts+1) adds 2.51 to net_rating. Because of the log scale, this means the benefit of scoring diminishes as points increase — going from 5 to 10 points per game has a larger net_rating impact than going from 20 to 25.
- **pts_usg / interaction term (coef = 0.94, p < 0.001):** When a player scores heavily AND has high usage, the net effect on team performance becomes positive. This interaction captures efficient high-usage players — those who justify their ball dominance with production.
- **usg_pct (coef = −46.97, p < 0.001):** On its own, high usage is strongly negative. However, this must be read together with the interaction term above. The combined effect of usg_pct and pts_usg determines the true impact of usage on net_rating.
- **ast (coef = 0.25, p < 0.001):** Assists remain consistently positive across all three models, reinforcing their genuine contribution to team performance.
- **reb (coef = 0.15, p < 0.001):** Rebounds are positive and significant, consistent with Models 1 and 2.
- **const (p = 0.192):** The intercept is not statistically significant, meaning when all predictors are zero the baseline net_rating is not meaningfully different from zero. This does not affect model validity.

**Why this is the final model:** Model 3 has the lowest condition number (best multicollinearity), the lowest AIC among Models 2 and 3, and is the only model grounded in a theoretical transformation. The log(pts) and interaction term reflect real basketball dynamics that raw linear models cannot capture.